In [4]:
import json
import numpy as np
import torch

from transformers import AutoTokenizer, AutoModelForMaskedLM
from sentence_transformers import SentenceTransformer
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"


In [ ]:
DATA_PATH = "/home/simonettos/thijs/classification/classification/datasets/tram__split_train_eda.json"
with open(DATA_PATH, "r", encoding="utf-8") as f:
    col = json.load(f)

# Convert column-oriented dict-of-dicts -> list of row dicts
row_ids = sorted(col["sentence"].keys(), key=lambda x: int(x) if x.isdigit() else x)

rows = []
for rid in row_ids:
    r = {"_id": rid}
    for k, v in col.items():
        r[k] = v.get(rid, None)
    rows.append(r)

len(rows), rows[0].keys()


(15358, dict_keys(['_id', 'sentence', 'labels', 'doc_title']))

In [7]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MLM_MODEL = "ehsanaghaei/SecureBERT"
SBERT_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"

tokenizer = AutoTokenizer.from_pretrained(MLM_MODEL)
mlm = AutoModelForMaskedLM.from_pretrained(MLM_MODEL).to(DEVICE)
mlm.eval()

sbert = SentenceTransformer(SBERT_MODEL, device=DEVICE)

MASK = tokenizer.mask_token
MASK_ID = tokenizer.mask_token_id
MASK, MASK_ID, DEVICE


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

('<mask>', 50264, 'cuda')

In [8]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)

def mlm_topk_predictions(masked_sentence: str, topk: int = 5, max_len: int = 256):
    """Return list of (token_str, prob) for the [MASK] position."""
    enc = tokenizer(masked_sentence, return_tensors="pt", truncation=True, max_length=max_len).to(DEVICE)
    input_ids = enc["input_ids"]
    mask_pos = (input_ids == MASK_ID).nonzero(as_tuple=False)
    if mask_pos.numel() == 0:
        return []

    pos = int(mask_pos[0, 1].item())
    with torch.no_grad():
        logits = mlm(**enc).logits  # [1, L, V]
        probs = torch.softmax(logits[0, pos, :], dim=-1)
        top = torch.topk(probs, k=topk)

    ids = top.indices.detach().cpu().numpy().tolist()
    ps  = top.values.detach().cpu().numpy().tolist()

    out = []
    for tid, p in zip(ids, ps):
        tok = tokenizer.convert_ids_to_tokens(int(tid))
        out.append((tok, float(p)))
    return out

def replace_word_with_mask(sentence: str, i: int):
    """Whitespace word masking, like the paper describes."""
    words = sentence.split()
    if i < 0 or i >= len(words):
        return None
    words[i] = MASK
    return " ".join(words)

def inject_token(masked_sentence: str, pred_token: str):
    """
    Replace MASK with pred_token. Handle basic BERT wordpieces.
    This is a simple approach suitable for notebook reproducibility.
    """
    parts = masked_sentence.split()
    if MASK not in parts:
        return None
    idx = parts.index(MASK)

    if pred_token.startswith("##"):
        if idx == 0:
            return None
        parts[idx - 1] = parts[idx - 1] + pred_token[2:]
        parts.pop(idx)
    else:
        parts[idx] = pred_token

    return " ".join(parts).replace(" ##", "")


In [20]:
def ttpxhunter_augment_sentence(sentence: str, theta: float = 0.975, topk: int = 5, max_len: int = 256):
    # original embedding
    orig_emb = sbert.encode([sentence], convert_to_numpy=True)[0]

    words = sentence.split()
    kept = []

    for i in range(len(words)):
        masked = replace_word_with_mask(sentence, i)
        if masked is None:
            continue

        preds = mlm_topk_predictions(masked, topk=topk, max_len=max_len)
        if not preds:
            continue

        for tok, prob in preds:
            cand = inject_token(masked, tok)
            if cand is None:
                continue

            cand_emb = sbert.encode([cand], convert_to_numpy=True)[0]
            sim = cosine_sim(orig_emb, cand_emb)

            if sim >= theta:
                kept.append({
                    "augmented_sentence": cand,
                    "masked_index": i,
                    "replacement": tok,
                    "mlm_prob": prob,
                    "similarity": sim
                })

    return kept
def has_T1_label(labels):
    return isinstance(labels, list) and any(l.startswith("T1") for l in labels)
def rows_to_column_json(rows):
    cols = {}
    for r in rows:
        rid = r["_id"]
        for k, v in r.items():
            if k not in cols:
                cols[k] = {}
            cols[k][rid] = v
    return cols


In [19]:
THETA = 0.975
TOPK = 5
from tqdm import tqdm
augmented_rows = []
num_aug = 0
num_considered = 0

for r in tqdm(rows):
    sentence = r.get("sentence", "")
    labels = r.get("labels", [])

    # --- FILTER: only sentences with labels starting with 'T1'
    if not isinstance(sentence, str) or not sentence.strip():
        continue
    if not has_T1_label(labels):
        continue

    num_considered += 1

    augs = ttpxhunter_augment_sentence(
        sentence,
        theta=THETA,
        topk=TOPK
    )

    for a in augs:
        new_r = dict(r)
        new_r["sentence"] = a["augmented_sentence"]
        new_r["_aug"] = {
            "method": "ttpxhunter_securebert_mlm",
            "theta": THETA,
            "topk": TOPK,
            "masked_index": a["masked_index"],
            "replacement": a["replacement"],
            "mlm_prob": a["mlm_prob"],
            "similarity": a["similarity"],
            "source_sentence": sentence,
        }
        augmented_rows.append(new_r)

    num_aug += len(augs)

print("Sentences considered (T1 only):", num_considered)
print("Augmented rows generated:", num_aug)


  0%|          | 74/15358 [00:11<38:08,  6.68it/s]  


KeyboardInterrupt: 

In [ ]:
all_rows = rows + augmented_rows
col_json = rows_to_column_json(all_rows)

OUT_PATH_COL = "/home/simonettos/thijs/classification/classification/datasets/tram_train_ttp_hunter.json"
with open(OUT_PATH_COL, "w", encoding="utf-8") as f:
    json.dump(col_json, f, indent=2, ensure_ascii=False)

print("Saved column-oriented JSON to:", OUT_PATH_COL)


[{'augmented_sentence': 'We cover topics such as domain fronting, SOCKS proxy, C2 traffic, Sigma rules, JARM, JA3/S, RITA & more.', 'masked_index': 0, 'replacement': 'We', 'mlm_prob': 0.5339601039886475, 'similarity': 1.0}, {'augmented_sentence': 'They cover topics such as domain fronting, SOCKS proxy, C2 traffic, Sigma rules, JARM, JA3/S, RITA & more.', 'masked_index': 0, 'replacement': 'They', 'mlm_prob': 0.2705085277557373, 'similarity': 0.991838276386261}, {'augmented_sentence': 'These cover topics such as domain fronting, SOCKS proxy, C2 traffic, Sigma rules, JARM, JA3/S, RITA & more.', 'masked_index': 0, 'replacement': 'These', 'mlm_prob': 0.03940068557858467, 'similarity': 0.9882773160934448}, {'augmented_sentence': 'I cover topics such as domain fronting, SOCKS proxy, C2 traffic, Sigma rules, JARM, JA3/S, RITA & more.', 'masked_index': 0, 'replacement': 'I', 'mlm_prob': 0.02578822337090969, 'similarity': 0.9924560189247131}, {'augmented_sentence': 'It cover topics such as domai

In [ ]:
We cover topics such as domain fronting, SOCKS proxy, C2 traffic, Sigma rules, JARM, JA3/S, RITA & :
We cover topics such as domain fronting, SOCKS proxy, C2 traffic, Sigma rules, JARM, JA3/S, RITA & more.

SyntaxError: invalid syntax (2861130561.py, line 1)